# Библтотеки

In [1]:
import json
import asyncio
import random
import openai
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Optional
from tqdm.asyncio import tqdm as atqdm

# Загрузка датасета

In [2]:
!git clone https://github.com/lorafei/Explainable_GEC.git

Cloning into 'Explainable_GEC'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 65 (delta 14), reused 65 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 5.44 MiB | 8.87 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Explainable_GEC
cfgs  pic	     README.md	      run.py		  utils
data  read_jsonl.py  requirement.txt  simpletransformers


In [2]:
data_path = Path("/content/Explainable_GEC/data/json/train.json")

In [3]:
rows = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

In [4]:
df.head()

,target,source,evidence_index,correction_index,error_type,predicted_parsing_order,origin
0,"[It, has, a, high, -, density, population, bec...","[It, has, a, high, -, density, population, bec...","[7, 9, 10, 11, 21, 23, 24, 25]","[8, 22]",Preposition,"{'1': 3, '5': 2, '7': 2, '8': 1, '9': 3, '10':...",A
1,"[Although, [NONE], it, is, an, industrial, cit...","[Although, of, it, is, an, industrial, city, ,...","[0, 17]","[1, 18]",Preposition,"{'1': 1, '18': 1}",A
2,"[Although, it, is, an, industrial, city, ,, th...","[Despite, it, is, an, industrial, city, ,, the...",[],"[0, 16]",Collocation,"{'0': 1, '5': 2, '8': 3, '16': 1, '21': 2, '24...",A
3,"[There, is, a, commercial, zone, along, the, w...","[There, are, a, commercial, zone, along, the, ...","[2, 3, 4, 50, 51, 52]","[1, 49]",Subject-Verb Agreement,"{'0': 2, '1': 1, '2': 3, '3': 2, '4': 3, '20':...",A
4,"[There, is, a, commercial, zone, along, the, w...","[There, are, a, commercial, zone, along, the, ...","[2, 3, 4, 50, 51, 52]","[1, 49]",Subject-Verb Agreement,"{'0': 2, '1': 1, '2': 3, '3': 2, '4': 3, '20':...",A


# Конфиг

In [ ]:
YANDEX_CLOUD_FOLDER = ""
YANDEX_CLOUD_API_KEY = ""
YANDEX_CLOUD_MODEL   = "qwen3-235b-a22b-fp8/latest"

TARGET_SAMPLES = 3000   
CONCURRENCY = 5        
SLEEP_AFTER_REQUEST = 2.0 
OUTPUT_PATH      = Path("ft_dataset.jsonl")

TASK_TYPES = [
    "grammar_choice",
    "transformation",
    "vocabulary_fill",
    "functional_matching",
]

# Клиент

In [6]:
async_client = openai.AsyncOpenAI(
    api_key  = YANDEX_CLOUD_API_KEY,
    base_url = "https://ai.api.cloud.yandex.net/v1",
    project  = YANDEX_CLOUD_FOLDER
)

# Промпты

In [7]:
SYSTEM_PROMPT = """
You are an expert English exercise generator.

Your task:
Generate exactly ONE exercise based on the user's grammar error and the requested task type.

TAXONOMY:
- Adjective Form
- Noun Inflection
- Noun Number
- Tense Usage
- Verb Form
- Verb Inflection
- Verb Agreement
- Verb Tense
- Part of Speech
- Adverb
- Determiner
- Particle
- Preposition
- Spelling
- Word Order
- No Error
(Use only these labels.)

Output requirements:
- JSON only.
- Strict adherence to the requested "task_type".
- NO bundles. Just a single exercise task.
- Instructions and content in English.

Safety:
- No PII.
- Correct English grammar in examples.
"""

In [8]:
USER_PROMPT_TEMPLATE = """
Generate exactly one exercise task matching the requested type.

INPUT DATA:
user_id: "{user_id}"
message_content: "{message_content}"
grammar_error: "{grammar_error}"
explanation: "{explanation}"
llm_confidence: {llm_confidence}
task_type: "{task_type}"

OUTPUT JSON SCHEMA (strict; JSON only):
{{
  "target_error_category": "string",
  "corrected_sentence": "string",
  "task": {{
      "type": "string",               # One of: vocabulary_fill, matching, grammar_choice, transformation, reading_tf, functional_match, writing_sample
      "instruction_en": "string",     # Task instruction in English
      "content_en": {{
        "context_text": "string" | null,
        "items": [
          {{
            "question_en": "string",
            "options_en": ["string"] | null,
            "student_answer_en": "string"
          }}
        ],
        "word_bank": ["string"] | null
      }}
  }}
}}


EXERCISE RULES (Strict Formatting):
1. Vocabulary Fill (Long text):
   - Generate a coherent text (email/story) with 12-20 gaps.
   - Provide a "word_bank" list.
   - In "items", break the text into logical segments or put full text in context_text with **_answer_** filled in.
   - word_bank must contain UNIQUE words only, no duplicates.
   - If a word is used multiple times, still list it once in word_bank.

2. Matching / Categories:
   - Provide 8-12 pairs or 10-16 category items.
   - Format student_answer_en as "**_1–D_**" or "**_Furniture: Sofa_**".

3. Grammar Choice (10-14 items):
   - MINIMUM 10 items, MAXIMUM 14 items. Never fewer than 10.
   - Provide exactly 2-3 options per item in parentheses.
   - student_answer_en MUST be the FULL sentence with the correct word bolded AND options shown.
   - Format: "Full sentence with **_correct_** (option1/option2/option3)."
   - Example: "The audience would boo and throw fruit **_at_** (to/at/toward) the actors."
   - NEVER return just the word alone like "at". Always full sentence.
   - The question_en must show the blank as ___ (three underscores), NOT pre-filled.
   - student_answer_en is ALWAYS the full sentence with bold answer + options. No exceptions.

4. Transformations (8-12 items):
   - MINIMUM 8 items. Never fewer than 8. Generate new sentences if needed.
   - Show the transformation goal (e.g. "Correct the verb tense").
   - student_answer_en MUST be the FULL rewritten sentence with bolded changes.
   - NEVER return just the changed word alone. Always the complete sentence.
   - Example: "She **_goes_** to the library every morning before class."

5. Reading (True/False):
   - Provide text (120-180 words) in context_text.
   - 8 statements in items.
   - Format student_answer_en as "**_True_**".

6. Functional Matching:
   - 8-10 items.
   - Each item MUST have options_en with 4 labelled choices (e.g. ["A. ...", "B. ...", "C. ...", "D. ..."]).
   - student_answer_en MUST be "**_1–C_**" format showing item number and correct letter.
   - Never set options_en to null for this task type.

7. Writing Sample:
   - Provide context_text (prompt).
   - In items, provide a "Student Answer" (60-90 words).
   - Use **_bold_** for key target grammar/vocab inside the student answer.

GENERAL RULES:
- Target the specific `grammar_error` primarily, but expand context to the general topic of the user's message.
- If `llm_confidence` < 0.75, assume the error might be a false positive and generate a "Verification/Choice" task first.
- Return ONLY valid JSON.
"""

# Утилиты

In [ ]:
def detokenize(tokens) -> str:
    """'It,has,a,...' → 'It has a ...'"""
    no_space = {'.', ',', ':', ';', '!', '?', ')', ']', '}', "n't"}

    if isinstance(tokens, list):
        token_list = [str(t).strip() for t in tokens]
    else:
        token_list = [t.strip() for t in str(tokens).split(',')]

    result = []
    for i, tok in enumerate(token_list):
        if tok == '[NONE]':
            continue
        if i == 0 or tok in no_space or tok.startswith("'"):
            result.append(tok)
        else:
            result.append(' ' + tok)
    return ''.join(result)


def make_explanation(source: str, target: str, error_type: str) -> str:
    """Простое автоматическое объяснение ошибки."""
    return (
        f"The sentence contains a '{error_type}' error. "
        f"The incorrect version is: \"{source}\". "
        f"The correct version is: \"{target}\"."
    )


def row_to_batch_item(row: pd.Series, task_type: str) -> Dict[str, Any]:
    source = detokenize(row['source'])
    target = detokenize(row['target'])
    error  = row['error_type']

    return {
        "user_id":         f"u_{row.name}",      
        "message_content": source,
        "grammar_error":   error,
        "explanation":     make_explanation(source, target, error),
        "llm_confidence":  0.95,
        "task_type":       task_type,
        "_source":         source,
        "_target":         target,
        "_record_id":      f"{row.name}_{task_type}",  
    }


async def generate_exercise_async(item, semaphore, retries=3):
    user_prompt = USER_PROMPT_TEMPLATE.format(
        user_id         = item["user_id"],
        message_content = item["message_content"],
        grammar_error   = item["grammar_error"],
        explanation     = item["explanation"],
        llm_confidence  = item["llm_confidence"],
        task_type       = item["task_type"],
    )
    async with semaphore:
        for attempt in range(retries + 1):
            try:
                response = await async_client.chat.completions.create(
                    model       = f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
                    temperature = 0.3,
                    messages    = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_prompt},
                    ],
                    max_tokens = 4000,
                )
                raw = response.choices[0].message.content

                if not raw or not raw.strip():
                    raise ValueError("Empty response from API")

                raw = raw.strip()
                if raw.startswith("```"):
                    raw = raw.split("```")[1]
                    if raw.startswith("json"):
                        raw = raw[4:]

                result = json.loads(raw)
                await asyncio.sleep(SLEEP_AFTER_REQUEST)
                return result

            except openai.RateLimitError:
                wait = 10 * (attempt + 1)  
                print(f"  ⏳ Rate limit, waiting {wait}s...")
                await asyncio.sleep(wait)
                continue
            except (json.JSONDecodeError, ValueError) as e:
                if attempt < retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                print(f"  ✗ [{item['user_id']}] failed after {retries+1} attempts: {e}")
                return None
            except Exception as e:
                if attempt < retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                print(f"  ✗ [{item['user_id']}] {e}")
                return None

def build_ft_record(item: Dict[str, Any], exercise: Dict[str, Any]) -> Dict[str, Any]:
    """
    Формирует одну запись для fine-tune датасета.
    Формат: {"prompt": "...", "completion": "..."}
    """
    user_prompt = USER_PROMPT_TEMPLATE.format(
        user_id        = item["user_id"],
        message_content= item["message_content"],
        grammar_error  = item["grammar_error"],
        explanation    = item["explanation"],
        llm_confidence = item["llm_confidence"],
        task_type      = item["task_type"],
    )
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": json.dumps(exercise, ensure_ascii=False)},
        ],
        "_meta": {
            "record_id":  item["_record_id"],
            "source":     item["_source"],
            "target":     item["_target"],
            "error_type": item["grammar_error"],
            "task_type":  item["task_type"],
        }
    }

async def process_item(item, semaphore, fout, lock, counter):
    exercise = await generate_exercise_async(item, semaphore)

    async with lock:
        if exercise is None:
            counter["errors"] += 1
            with open("ft_dataset_errors.txt", "a") as ferr:
                ferr.write(item["_record_id"] + "\n")
        else:
            record = build_ft_record(item, exercise)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            fout.flush()
            counter["saved"] += 1

# Разметка

In [ ]:
async def generate_dataset_async(df: pd.DataFrame, target_n: int):
    done_ids = set()
    if OUTPUT_PATH.exists():
        with OUTPUT_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done_ids.add(r["_meta"]["record_id"])
                except:
                    pass
        print(f"Уже готово: {len(done_ids)} записей, продолжаем...")

    indices = list(range(len(df)))
    if target_n > len(indices):
        indices = indices * (target_n // len(indices) + 1)
    random.shuffle(indices)
    indices = indices[:target_n]

    task_cycle = TASK_TYPES * (target_n // len(TASK_TYPES) + 1)

    items = []
    for i, idx in enumerate(indices):
        row = df.iloc[idx]
        item = row_to_batch_item(row, task_cycle[i])
        if item["_record_id"] not in done_ids:
            items.append(item)

    print(f"Осталось сгенерировать: {len(items)}")

    semaphore = asyncio.Semaphore(CONCURRENCY)
    lock      = asyncio.Lock()
    counter   = {"saved": len(done_ids), "errors": 0, "total": target_n}

    with OUTPUT_PATH.open("a", encoding="utf-8") as fout:
        tasks = [
            process_item(item, semaphore, fout, lock, counter)
            for item in items
        ]
        await atqdm.gather(*tasks, total=len(items), desc="Generating")

    print(f"\n✓ Готово: {counter['saved']} записей, {counter['errors']} ошибок → {OUTPUT_PATH}")

In [11]:
OUTPUT_PATH.unlink(missing_ok=True)
await generate_dataset_async(df, target_n=TARGET_SAMPLES)

Осталось сгенерировать: 3000


Generating: 100%|██████████| 3000/3000 [5:39:13<00:00,  6.78s/it]


✓ Готово: 3000 записей, 0 ошибок → ft_dataset.jsonl


In [11]:
async def debug_single():
    response = await async_client.chat.completions.create(
        model       = f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
        temperature = 0.3,
        messages    = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",   "content": "Say hello in JSON format: {\"message\": \"hello\"}"},
        ],
        max_tokens = 100,
    )
    print("finish_reason:", response.choices[0].finish_reason)
    print("content:", repr(response.choices[0].message.content))
    print("full response:", response)

await debug_single()

finish_reason: stop
content: '{"message": "hello"}'
full response: ChatCompletion(id='chatcmpl-1396dec9-433d-42f7-ba3e-b3fcdb83e135', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"message": "hello"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773140104, model='gpt://qwen3-235b-a22b-fp8/latest', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=7, prompt_tokens=31, total_tokens=38, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)))


In [19]:
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        print(f"\n{'='*60}")
        print(f"ЗАПИСЬ #{i+1}")
        print(f"{'='*60}")
        print(f"META: error={record['_meta']['error_type']} | task={record['_meta']['task_type']}")
        print(f"SOURCE: {record['_meta']['source']}")
        print(f"TARGET: {record['_meta']['target']}")
        print(f"\nASSISTANT OUTPUT:")
        exercise = json.loads(record['messages'][2]['content'])
        print(json.dumps(exercise, indent=2, ensure_ascii=False))


ЗАПИСЬ #1
META: error=Others | task=functional_matching
SOURCE: Many people say that public transport is not comfortable. that s true. From my point of view, a bus is not so uncomfortable.
TARGET: Many people say that public transport is not comfortable. That's true. From my point of view, a bus is not so uncomfortable.

ASSISTANT OUTPUT:
{
  "target_error_category": "Others",
  "corrected_sentence": "Many people say that public transport is not comfortable. That's true. From my point of view, a bus is not so uncomfortable.",
  "task": {
    "type": "functional_match",
    "instruction_en": "Match each sentence fragment on the left with the correct continuation on the right that makes a grammatically correct and meaningful sentence. Pay attention to punctuation, contractions, and sentence structure.",
    "content_en": {
      "context_text": null,
      "items": [
        {
          "question_en": "1. Many people say that public transport is not comfortable.",
          "options_en"